# Smoke test - Kippenhahn dataset builder

Quick end-to-end test on a reduced config, before generating the "real" dataset.
Verifies that the full pipeline (generation -> split -> augmentation -> features -> standardization)
runs without errors and produces sensible shapes/distributions.

In [1]:
import numpy as np

from dataset_config import DatasetConfig
from dataset_builder import (
    build_base_pool,
    split_base_pool,
    augment_train_split,
    extract_features_for_splits,
    standardize,
    sanity_check,
    build_final_dataset,
    save_dataset,
)

## 1. Reduced config

Small numbers, only to verify that the pipeline doesn't blow up: nothing to do
with the numbers you'll use for the final dataset.

In [2]:
config = DatasetConfig(
    n_total_core=200,
    seed=0,
)
config

DatasetConfig(n_total_core=200, case_proportions={1: 0.15, 2: 0.25, 3: 0.1, 4: 0.5}, n_augment_per_case={1: 5, 2: 5, 3: 8, 4: 5}, n_hard_per_case=200, train_fraction=0.8, seed=0)

## 2. Step-by-step pipeline (to inspect intermediate shapes)

In [3]:
A_all, labels_all, meta_all = build_base_pool(config)
print("A_all:", A_all.shape, A_all.dtype)
print("labels_all:", labels_all.shape)
print("class distribution (pre-split, pre-augmentation):", np.bincount(labels_all)[1:])
print("len(meta_all):", len(meta_all))

assert A_all.shape[0] == labels_all.shape[0] == len(meta_all)

A_all: (200, 3, 3) complex128
labels_all: (200,)
class distribution (pre-split, pre-augmentation): [ 30  50  20 100]
len(meta_all): 200


In [4]:
splits = split_base_pool(A_all, labels_all, meta_all, config)
for name in ("train", "val"):
    y = splits[name]["y"]
    print(f"{name}: n={len(y)}, class distribution={np.bincount(y)[1:]}")

train: n=160, class distribution=[24 40 16 80]
val: n=40, class distribution=[ 6 10  4 20]


In [5]:
splits = augment_train_split(splits, config)

n_val = len(splits["val"]["y"])  # sanity: val must not be touched
print("train after augmentation:", splits["train"]["A"].shape)
print("train class distribution (post augmentation):", np.bincount(splits["train"]["y"])[1:])
print("val unchanged:", splits["val"]["A"].shape)

train after augmentation: (1008, 3, 3)
train class distribution (post augmentation): [144 240 144 480]
val unchanged: (40, 3, 3)


In [6]:
splits = extract_features_for_splits(splits)

for name in ("train", "val"):
    X = splits[name]["X"]
    print(f"{name}: X.shape={X.shape}")
    assert X.shape[1] == 9, "expected 9 features (10 coefficients - 1 constant)"
    assert not np.isnan(X).any(), "NaN in features"
    assert not np.isinf(X).any(), "inf in features"

train: X.shape=(1008, 9)
val: X.shape=(40, 9)


In [7]:
scaler = standardize(splits)

X_train = splits["train"]["X"]
print("post-standardization mean (train, expected ~0):", X_train.mean(axis=0).round(6))
print("post-standardization std (train, expected ~1):", X_train.std(axis=0).round(6))
print("scaler mean:", scaler["mean"])
print("scaler std:", scaler["std"])

post-standardization mean (train, expected ~0): [ 0. -0.  0.  0.  0.  0.  0. -0. -0.]
post-standardization std (train, expected ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1.]
scaler mean: [ 0.00405004 -0.00377199 -0.04626975 -0.00628559 -0.03564749  0.03696427
  0.00388313 -0.1420448  -0.04594252]
scaler std: [0.03409033 0.05494116 0.19809037 0.05444545 0.24450978 0.59673116
 0.04298428 0.15636019 0.51362023]


## 3. Full orchestrator test (`build_final_dataset`)

Same result as above, but through the single function that will be used in production.

In [8]:
dataset = build_final_dataset(config)
sanity_check(dataset)

## 4. Anti-leakage check (train vs val)

Heuristic check: no row of `X_val` should be nearly identical to a row of `X_train`
(this could happen if, by mistake, augmentation leaked into val or if the split
mixed augmented copies between the two groups).

In [9]:
X_train = dataset["train"]["X"]
X_val = dataset["val"]["X"]

# minimum distance of each val point from the closest train point
dists = np.linalg.norm(X_val[:, None, :] - X_train[None, :, :], axis=2)
min_dists = dists.min(axis=1)

print("minimum distance val->train: min={:.4f}, median={:.4f}".format(
    min_dists.min(), np.median(min_dists)
))
# note: indicative threshold, adjust based on the actual scale of standardized features
n_suspicious = (min_dists < 1e-6).sum()
print(f"val points nearly identical to a train point: {n_suspicious} / {len(min_dists)}")

minimum distance val->train: min=0.6485, median=1.3599
val points nearly identical to a train point: 0 / 40


## 5. Trial save (on a small config, only to test I/O)

In [10]:
save_dataset(dataset, "kippenhahn_smoke_test")

Salvato: kippenhahn_smoke_test_arrays.npz, kippenhahn_smoke_test_meta.pkl
